In [13]:
# scripts/make_eda.py
"""
EDA v2 — understands the new 2025-style export (tags, institutionCountry, isITC)
and produces the shape that make_map.py and make_dashboard.py already expect.

Hard-coded default input:
- data/raw/COST_Action_CA23107_Contacts_export_05-11-2025.xlsx

Outputs
-------
- data/raw/data_raw.csv
- data/raw/data_raw_<excelname>.csv
- data/processed/data_clean.csv
- outputs/eda_overview.csv
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Dict, Any
import argparse
import re
import sys

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Repo root detection
# -----------------------------------------------------------------------------
def _find_repo_root() -> Path:
    try:
        here = Path(__file__).resolve()
        return here.parent.parent
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd


ROOT     = _find_repo_root()
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUT_DIR  = ROOT / "outputs"
AUX_DIR  = ROOT / "data"

PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root →", ROOT)

# hard-coded file you wanted
HARDCODED_XLSX = RAW_DIR / "COST_Action_CA23107_Contacts_export_05-11-2025.xlsx"


# -----------------------------------------------------------------------------
# generic helpers
# -----------------------------------------------------------------------------
def write_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved → {path.relative_to(ROOT)}")
    return path


def snake_case(name: str) -> str:
    name = str(name).replace("\u00A0", " ")
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = " ".join(name.split()).strip().lower().replace("-", " ")
    return re.sub(r"\s+", "_", name)


def clean_text_series(s: pd.Series) -> pd.Series:
    out = (
        s.astype(str)
         .str.replace(r"\*", "", regex=True)
         .str.replace("\u00A0", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )
    return out.replace({"": np.nan, "nan": np.nan})


# columns that usually are Y/N-ish
_YN_PATTERNS = [re.compile(p, re.I) for p in (
    r"^wg[\s_]*\d+",
    r"^is_",
    r"\bmc\b|\bmc_member\b",
    r"\bcore\b|\bcore_group\b",
    r"^wg_member$",
)]


def looks_like_yn(names: Iterable[str]) -> list[bool]:
    return [any(p.search(str(c).lower()) for p in _YN_PATTERNS) for c in names]


def status_from_tokens(s: pd.Series) -> pd.Series:
    raw = s.astype(str)
    stripped = raw.str.strip()
    low = stripped.str.lower()

    # non-capturing group to avoid warning
    pend_mask = low.str.contains(r"\b(?:pending|tbc|awaiting)\b", na=False)

    token = (
        low.str.replace(r"[()\[\]{}.,;:!/?\-]+", " ", regex=True)
           .str.replace(r"\s+", " ", regex=True)
           .str.strip()
    )

    YES = {"y", "yes", "true", "1", "member", "x"}
    NO  = {"n", "no", "false", "0"}

    yes_mask = token.isin(YES)
    no_mask  = token.isin(NO)

    out = pd.Series(index=s.index, dtype="object")
    out[yes_mask & ~pend_mask] = "Yes"
    out[no_mask  & ~pend_mask] = "No"
    out[pend_mask]             = "Pending"

    others = out.isna()
    out[others] = stripped[others].where(stripped[others].ne(""), np.nan)

    return out.map(lambda v: v.title() if isinstance(v, str) else v)


def status_to_nullable_bool(s: pd.Series) -> pd.Series:
    return s.map({"Yes": True, "No": False}).astype("boolean")


def gentle_type_infer(s: pd.Series) -> pd.Series:
    if s.dtype == "object":
        raw = s.astype(str).str.replace(",", "").str.strip()
        looks_num = raw.str.match(r"^-?\d+(\.\d+)?$", na=False)
        if looks_num.mean() >= 0.6:
            return pd.to_numeric(raw, errors="coerce")

    if s.dtype == "object" or s.dtype.kind in "Mm":
        if s.dtype.kind in "Mm":
            return s
        sample = s.astype(str).str.lower()
        looks_date = sample.str.contains(
            r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
            regex=True, na=False
        )
        if looks_date.mean() >= 0.5:
            parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
            if parsed.notna().mean() >= 0.5:
                return parsed
    return s


# -----------------------------------------------------------------------------
# non-country stoplist
# -----------------------------------------------------------------------------
NON_COUNTRY_LC = {
    "european rtd organisations",
    "european rtd organizations",
    "european commission and eu agencies",
    "european commission",
    "european union",
    "eu",
}


def clean_country_column(df: pd.DataFrame) -> pd.DataFrame:
    matches = [c for c in df.columns if c.lower() in ("country", "institutioncountry")]
    if not matches:
        return df
    col = matches[0]

    base = df[col].astype(str).str.replace(r"\(.*?\)", "", regex=True)
    base = clean_text_series(base)
    low = base.str.lower().str.strip()

    aliases = {
        "uk": "United Kingdom", "united kingdom": "United Kingdom", "great britain": "United Kingdom",
        "czech republic": "Czechia",
        "turkey": "Türkiye", "turkiye": "Türkiye",
        "north macedonia": "North Macedonia", "macedonia": "North Macedonia",
        "bosnia and herzegovina": "Bosnia and Herzegovina",
        "bosnia & herzegovina": "Bosnia and Herzegovina",
        "moldova": "Moldova",
        "ivory coast": "Côte d'Ivoire", "cote d ivoire": "Côte d'Ivoire",
        "republic of kosovo": "Kosovo", "kosovo*": "Kosovo", "kosovo": "Kosovo",
        "republic of serbia": "Serbia",
        "republic of north macedonia": "North Macedonia",
    }

    df["country_clean"] = base

    alias_mask = low.isin(aliases)
    df.loc[alias_mask, "country_clean"] = low.map(aliases)

    # set umbrella/eu-ish to NA
    non_country_mask = low.isin(NON_COUNTRY_LC)
    df.loc[non_country_mask, "country_clean"] = pd.NA

    return df


def value_counts_preview(s: pd.Series, n: int = 10) -> str:
    vc = s.astype("string").fillna("<NA>").value_counts(dropna=False).head(n)
    return "; ".join(f"{k}: {int(v)}" for k, v in vc.items())


def mark_section(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "section", name)
    return out


# -----------------------------------------------------------------------------
# ITC helpers
# -----------------------------------------------------------------------------
def _norm_country(s: str) -> str:
    s = (str(s) or "").strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    s = s.replace("’", "'")
    s = " ".join(s.split()).lower()
    s = s.replace("republic of ", "")
    aliases_lc = {
        "czech republic": "czechia",
        "macedonia": "north macedonia",
        "turkey": "türkiye", "turkiye": "türkiye", "türkiye": "türkiye",
        "uk": "united kingdom",
        "cote d'ivoire": "côte d'ivoire",
    }
    return aliases_lc.get(s, s)


def load_itc_set(path: Path) -> set[str]:
    raw = path.read_text(encoding="utf-8").splitlines()
    items = [_norm_country(x) for x in raw if str(x).strip()]
    return set(items)


# -----------------------------------------------------------------------------
# TAG PARSER
# -----------------------------------------------------------------------------
TAG_SPLIT_RE = re.compile(r"[;,]")

# leadership words that should force core when attached to WG tags
WG_LEADERSHIP_KEYS = (
    "leader",
    "co-leader",
    "co leader",
    "chair",
    "vicechair",
    "vice-chair",
    "coordinator",
    "vice coordinator",
)


def _is_mc_tag(t: str) -> bool:
    tl = t.lower().strip()
    return (
        tl == "mc"
        or tl == "mc member"
        or tl.startswith("mc ")
        or tl.startswith("mc chair")
        or tl.startswith("mc vice")
    )


def _is_numbered_wg_tag(t: str) -> int | None:
    tl = t.lower()
    if "wg1" in tl or "wg 1" in tl or "working group 1" in tl:
        return 1
    if "wg2" in tl or "wg 2" in tl or "working group 2" in tl:
        return 2
    if "wg3" in tl or "wg 3" in tl or "working group 3" in tl:
        return 3
    if "wg4" in tl or "wg 4" in tl or "working group 4" in tl:
        return 4
    if "wg5" in tl or "wg 5" in tl or "working group 5" in tl:
        return 5
    return None


def parse_tags_to_flags(raw_tags: str) -> Dict[str, Any]:
    """
    Your rule, refined:
    - MC → mc_member = Yes
    - WG1..5 → wgX = Yes, wg_member = Yes
    - WG1..5 **with leadership keyword** → same as above **plus** core_group = Yes, and store the tag
    - any other tag → core_group = Yes, and store the tag
    """
    tags = [t.strip() for t in TAG_SPLIT_RE.split(raw_tags or "") if t.strip()]
    is_mc = False
    is_wg_member = False
    wg_flags = {1: False, 2: False, 3: False, 4: False, 5: False}
    core = False
    core_titles: list[str] = []

    for t in tags:
        t_low = t.lower()

        # MC family
        if _is_mc_tag(t):
            is_mc = True
            continue

        # numbered WG
        wg_num = _is_numbered_wg_tag(t)
        if wg_num is not None:
            wg_flags[wg_num] = True
            is_wg_member = True

            # extra rule: WG + leadership → also core
            if any(k in t_low for k in WG_LEADERSHIP_KEYS):
                core = True
                core_titles.append(t.strip())
            continue

        # WG-ish but not numbered → WG member + core
        if "working group" in t_low or t_low.startswith("wg ") or t_low.startswith("wg-"):
            is_wg_member = True
            core = True
            core_titles.append(t.strip())
            continue

        # anything else → core
        core = True
        core_titles.append(t.strip())

    return {
        "mc_member": "Yes" if is_mc else np.nan,
        "core_group": "Yes" if core else np.nan,
        "core_group_member_title": "; ".join(core_titles) if core_titles else np.nan,
        "wg_member": "Yes" if is_wg_member else np.nan,
        "wg1": "Yes" if wg_flags[1] else np.nan,
        "wg2": "Yes" if wg_flags[2] else np.nan,
        "wg3": "Yes" if wg_flags[3] else np.nan,
        "wg4": "Yes" if wg_flags[4] else np.nan,
        "wg5": "Yes" if wg_flags[5] else np.nan,
    }


def adapt_new_export_to_old_shape(df: pd.DataFrame) -> pd.DataFrame:
    cols = set(df.columns)

    if "country" not in cols and "institutioncountry" in cols:
        df["country"] = df["institutioncountry"]

    # debug bucket for WG leaders that still didn't get core_group
    suspicious_core: list[dict[str, str]] = []

    if "tags" in cols:
        tags_series = df["tags"].fillna("").astype(str)
        parsed_rows = []

        for idx, raw_tags in tags_series.items():
            parsed = parse_tags_to_flags(raw_tags)
            parsed_rows.append(parsed)

            # DEBUG: only shout when tag looks WG-ish AND leadership-y, but parser didn't set core
            if pd.isna(parsed["core_group"]) and raw_tags.strip():
                low = raw_tags.lower()
                if ("working group" in low or low.startswith("wg")) and any(k in low for k in WG_LEADERSHIP_KEYS):
                    suspicious_core.append({"index": str(idx), "tags": raw_tags})

        parsed_df = pd.DataFrame(parsed_rows)
        for col in parsed_df.columns:
            df[col] = parsed_df[col]

        if suspicious_core:
            print("\n[DEBUG] WG leadership tags that did NOT become core_group=Yes (tweak parser or source):")
            for item in suspicious_core[:25]:
                print("  - row", item["index"], "→", item["tags"])
            if len(suspicious_core) > 25:
                print(f"  ... and {len(suspicious_core) - 25} more")

    # ITC from file-level cols if present
    if "itc_countries" not in df.columns:
        for c in df.columns:
            if c.lower() in ("isitc", "is_itc", "isitc?", "isitc "):
                itc_norm = df[c].astype(str).str.strip().str.lower()
                df["itc_countries"] = np.where(itc_norm.isin(["yes", "y", "true", "1"]), "Yes", "No")
                break

    return df


# -----------------------------------------------------------------------------
# CLI
# -----------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Build data_clean.csv from old or new layout (tags-aware) without changing dashboards."
    )
    p.add_argument("--input", "-i", type=str, default=None,
                   help="Excel file (default = hard-coded COST_Action_CA23107_Contacts_export_05-11-2025.xlsx)")
    p.add_argument("--sheet", "-s", default=0,
                   help="Sheet index (0) or name")
    p.add_argument("--topn", type=int, default=25,
                   help="Top-N values for categorical previews")
    if "ipykernel" in sys.modules or "IPython" in sys.modules:
        args, _ = p.parse_known_args()
    else:
        args = p.parse_args()
    return args


# -----------------------------------------------------------------------------
# main
# -----------------------------------------------------------------------------
def main() -> None:
    args = parse_args()

    # use hard-coded unless user overrides
    if args.input:
        excel_path = Path(args.input)
        if not excel_path.is_absolute():
            excel_path = (ROOT / excel_path).resolve()
    else:
        excel_path = HARDCODED_XLSX

    if not excel_path.exists():
        print(f"ERROR: Excel not found: {excel_path}")
        sys.exit(1)

    try:
        rel = excel_path.relative_to(ROOT)
    except Exception:
        rel = excel_path
    print(f"\nReading Excel: {rel}")

    sheet  = args.sheet
    top_n  = args.topn
    df_raw = pd.read_excel(excel_path, sheet_name=sheet)

    # save raw copies
    write_csv(df_raw, RAW_DIR / "data_raw.csv")
    safe_stem = (
        excel_path.stem
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
    )
    write_csv(df_raw, RAW_DIR / f"data_raw_{safe_stem}.csv")

    # Y/N detection before cleaning
    yn_flags_before = looks_like_yn(df_raw.columns)
    yn_candidates_raw = [c for c, is_yn in zip(df_raw.columns, yn_flags_before) if is_yn]
    yn_before_df = pd.DataFrame({
        "original_name": yn_candidates_raw,
        "normalized_name_if_any": [snake_case(c) for c in yn_candidates_raw],
        "dtype_raw": [str(df_raw[c].dtype) for c in yn_candidates_raw],
        "top_values_preview": [value_counts_preview(df_raw[c], n=10) for c in yn_candidates_raw],
    })

    # start actual cleaning
    df = df_raw.copy()

    # header → snake_case
    old_to_new = {c: snake_case(c) for c in df.columns}
    df.rename(columns=old_to_new, inplace=True)

    # new export → old shape (creates mc/core/wg1..5/wg_member/core_group_member_title)
    df = adapt_new_export_to_old_shape(df)

    # clean text
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        df[obj_cols] = df[obj_cols].apply(clean_text_series)

    # country clean
    df = clean_country_column(df)

    # overlay maintained ITC list
    itc_path = AUX_DIR / "itc_countries.txt"
    base_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
    if itc_path.exists() and base_col:
        itc_set = load_itc_set(itc_path)
        df["itc_countries"] = df[base_col].map(lambda x: "Yes" if _norm_country(x) in itc_set else "No")
        print(f"Loaded ITC list ({len(itc_set)} names) → added 'itc_countries' (Yes/No).")
    elif base_col and "itc_countries" not in df.columns:
        df["itc_countries"] = "No"

    # pending-aware flags
    yn_flags_after = looks_like_yn(df.columns)
    yn_cols_after  = [c for c, is_yn in zip(df.columns, yn_flags_after) if is_yn]

    status_cols = []
    for c in yn_cols_after:
        sc = f"{c}_status"
        df[sc] = status_from_tokens(df[c])
        status_cols.append(sc)

    # unknown report
    unknown_report = {}
    for c in yn_cols_after:
        sc = f"{c}_status"
        st = df[sc].dropna()
        unknown_report[sc] = int((~st.isin(["Yes", "No", "Pending"])).sum())
    print("Unknown statuses (treated as <NA>):",
          {k: v for k, v in unknown_report.items() if v})

    # replace original flag columns with nullable booleans
    for c in yn_cols_after:
        sc = f"{c}_status"
        df[c] = status_to_nullable_bool(df[sc])

    # derive any_wg
    wg_cols = [c for c in yn_cols_after if re.match(r"(?i)^wg[\s_]*\d+", c)]
    if wg_cols:
        any_true = df[wg_cols].apply(lambda r: bool(r.fillna(False).any()), axis=1)
        wg_status_cols = [f"{c}_status" for c in wg_cols if f"{c}_status" in df.columns]
        if wg_status_cols:
            statuses = df[wg_status_cols]
            any_pending = statuses.eq("Pending").any(axis=1)
            known = statuses.isin(["Yes", "No", "Pending"]) | statuses.isna()
            any_unknown = ~known.all(axis=1)
        else:
            any_pending = pd.Series(False, index=df.index)
            any_unknown = pd.Series(False, index=df.index)

        df["any_wg"] = pd.Series(
            np.where(
                any_true,
                True,
                np.where(any_pending | any_unknown, pd.NA, False)
            ),
            dtype="boolean"
        )
    else:
        df["any_wg"] = pd.Series([False] * len(df), dtype="boolean")

    # assigned_working_groups column (pretty)
    wg_labels = {"wg1": "WG1", "wg2": "WG2", "wg3": "WG3", "wg4": "WG4", "wg5": "WG5"}
    def _assigned_wgs(row: pd.Series) -> str:
        parts = []
        for col, label in wg_labels.items():
            if col in row and bool(row[col] is True):
                parts.append(label)
        return "; ".join(parts) if parts else ""
    df["assigned_working_groups"] = df.apply(_assigned_wgs, axis=1)

    # conservative type inference
    skip_cols = set(status_cols + yn_cols_after + ["any_wg", "itc_countries"])
    for c in df.columns:
        if c in skip_cols:
            continue
        df[c] = gentle_type_infer(df[c])

    # write cleaned CSV (your map + dashboard read this)
    write_csv(df, PROC_DIR / "data_clean.csv")

    # build EDA long csv
    eda_parts: list[pd.DataFrame] = []

    dtype_changes = pd.DataFrame({
        "original_name": list(df_raw.columns),
        "cleaned_name": [old_to_new.get(c, c) for c in df_raw.columns],
        "dtype_raw": [str(df_raw[c].dtype) for c in df_raw.columns],
        "dtype_final": [str(df[old_to_new.get(c, c)].dtype) if old_to_new.get(c, c) in df.columns else "<missing>" for c in df_raw.columns],
    })
    dtype_changes["changed"] = dtype_changes["dtype_raw"] != dtype_changes["dtype_final"]
    eda_parts.append(mark_section(dtype_changes, "dtype_changes"))

    if not yn_before_df.empty:
        eda_parts.append(mark_section(yn_before_df, "yn_detection_before"))

    if yn_cols_after:
        rows = []
        for c in yn_cols_after:
            s  = df[c]
            sc = f"{c}_status"
            true_count  = int((s == True).sum())   # noqa: E712
            na_count    = int(s.isna().sum())
            non_na      = int(s.notna().sum())
            false_count = int(non_na - true_count)
            rows.append({
                "flag_col": c,
                "status_col": sc,
                "dtype_flag": str(s.dtype),
                "true_count": true_count,
                "false_count": false_count,
                "na_count": na_count,
                "status_preview": value_counts_preview(df[sc], n=10),
            })
        eda_parts.append(mark_section(pd.DataFrame(rows), "yn_after_pending_aware"))

    cols_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    }).sort_values("column")
    eda_parts.append(mark_section(cols_summary, "columns_summary"))

    num_cols = df.select_dtypes(include=[np.number, "boolean"]).columns
    if len(num_cols):
        desc_long = (
            df[num_cols].describe(include="all")
              .T.reset_index().rename(columns={"index": "column"})
              .melt(id_vars="column", var_name="metric", value_name="value")
        )
        eda_parts.append(mark_section(desc_long, "numeric_summary"))

    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    if len(cat_cols):
        rows = []
        for c in cat_cols:
            vc = df[c].astype("string").fillna("<NA>").value_counts(dropna=False).head(top_n)
            rows.extend({"column": c, "value": k, "count": int(v)} for k, v in vc.items())
        eda_parts.append(mark_section(pd.DataFrame(rows), "top_values"))

    country_col = [c for c in df.columns if c == "country_clean"] or [c for c in df.columns if c == "country"]
    if country_col:
        country_counts = (
            df[country_col[0]].astype("string").fillna("<NA>")
              .value_counts(dropna=False).rename_axis("country")
              .reset_index(name="count")
        )
        eda_parts.append(mark_section(country_counts, "countries_counts"))

    eda_overview = pd.concat(eda_parts, ignore_index=True, sort=False) if eda_parts else pd.DataFrame({"section": []})
    write_csv(eda_overview, OUT_DIR / "eda_overview.csv")

    print("\nAll files written under:")
    print(" -", RAW_DIR.relative_to(ROOT))
    print(" -", PROC_DIR.relative_to(ROOT))
    print(" -", OUT_DIR.relative_to(ROOT))


if __name__ == "__main__":
    main()


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood

Reading Excel: data\raw\COST_Action_CA23107_Contacts_export_05-11-2025.xlsx
Saved → data\raw\data_raw.csv
Saved → data\raw\data_raw_COST_Action_CA23107_Contacts_export_05_11_2025.csv
Loaded ITC list (25 names) → added 'itc_countries' (Yes/No).
Unknown statuses (treated as <NA>): {'mc_member_status': 156, 'core_group_status': 189, 'wg_member_status': 4, 'wg1_status': 82, 'wg2_status': 133, 'wg3_status': 112, 'wg4_status': 118, 'wg5_status': 129}
Saved → data\processed\data_clean.csv
Saved → outputs\eda_overview.csv

All files written under:
 - data\raw
 - data\processed
 - outputs
